In [ ]:
# Import required packages
# Import required packages
%pip install pandas numpy matplotlib seaborn requests python-dotenv newsapi-python vaderSentiment textblob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import requests
import json
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

print("Packages loaded successfully")

In [ ]:
import os
from newsapi import NewsApiClient
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from dotenv import load_dotenv

# Initialize VADER sentiment analyzer
analyzer = SentimentIntensityAnalyzer()

# 1. Dynamically build the absolute path to your file
home_dir = os.path.expanduser("~")
file_path = os.path.join(home_dir, "Downloads", "geopolitical-news-sentiment", "NewsAPI_Key.env")

# 2. Load the environment variables using the absolute path
if os.path.exists(file_path):
    load_dotenv(dotenv_path=file_path)
    print(f"Found key file at: {file_path}")
else:
    print(f"⚠️ Warning: Could not find the file at {file_path}")
    print("Please double-check your folder names and file spelling!")

# 3. Retrieve the API key securely from your system environment
news_api_key = os.getenv("NEWSAPI_KEY")

# 4. Check to ensure the key was loaded properly without exposing it
if not news_api_key:
    raise ValueError("API Key not found! Please check that 'NEWSAPI_KEY=...' is written correctly inside your file.")
else:
    print("✅ NewsAPI Key located and loaded securely!")

# 5. Initialize the NewsAPI client
newsapi = NewsApiClient(api_key=news_api_key)

# 6. Test API connection
try:
    top_headlines = newsapi.get_top_headlines(
        q="Russia sanctions",
        language="en"
    )
    articles = top_headlines.get('articles', [])
    print(f"\n--- API Test: Found {len(articles)} headlines for 'Russia sanctions' ---")
    for article in articles[:3]:
        print(f"- {article['title']}")
        
except Exception as e:
    print(f"An error occurred with NewsAPI: {e}")

# 7. Sentiment analyzer test
print("\n--- Sentiment Analyzer Test ---")
test_headline = "Russia launches major offensive amid escalating sanctions pressure"
score = analyzer.polarity_scores(test_headline)
print(f"Headline: {test_headline}")
print(f"Compound score: {score['compound']} (range: -1.0 to +1.0)")

In [ ]:
# Define our target countries and topics
targets = {
    'Russia': 'Russia sanctions war Ukraine',
    'Iran': 'Iran nuclear sanctions',
    'China': 'China trade economy Taiwan',
    'North Korea': 'North Korea missile nuclear',
    'Belarus': 'Belarus Lukashenko opposition'
}

# Pull headlines for each target
def fetch_headlines(query, days_back=30):
    from_date = (datetime.now() - timedelta(days=days_back)).strftime('%Y-%m-%d')
    try:
        response = newsapi.get_everything(
            q=query,
            from_param=from_date,
            language='en',
            sort_by='publishedAt',
            page_size=50
        )
        articles = response['articles']
        return [{
            'title': a['title'],
            'published': a['publishedAt'],
            'source': a['source']['name'],
            'description': a['description']
        } for a in articles if a['title'] != '[Removed]']
    except Exception as e:
        print(f"Error fetching {query}: {e}")
        return []

# Fetch all headlines
all_headlines = {}
for country, query in targets.items():
    headlines = fetch_headlines(query)
    all_headlines[country] = headlines
    print(f"{country}: {len(headlines)} headlines fetched")

In [ ]:
# Run sentiment analysis on all headlines
def analyze_sentiment(headlines, country):
    results = []
    for article in headlines:
        title = article['title'] or ''
        description = article['description'] or ''
        
        # Combine title and description for richer analysis
        text = f"{title}. {description}"
        
        # Get VADER sentiment scores
        scores = analyzer.polarity_scores(text)
        
        results.append({
            'country': country,
            'title': title,
            'source': article['source'],
            'published': pd.to_datetime(article['published']),
            'negative': scores['neg'],
            'neutral': scores['neu'],
            'positive': scores['pos'],
            'compound': scores['compound'],
            'sentiment_label': 'POSITIVE' if scores['compound'] >= 0.05 
                             else 'NEGATIVE' if scores['compound'] <= -0.05 
                             else 'NEUTRAL'
        })
    return results

# Process all countries
all_results = []
for country, headlines in all_headlines.items():
    results = analyze_sentiment(headlines, country)
    all_results.extend(results)

# Convert to dataframe
df = pd.DataFrame(all_results)
df = df.sort_values('published', ascending=False)

print(f"Total articles analyzed: {len(df)}")
print(f"\nSentiment breakdown:")
print(df['sentiment_label'].value_counts())
print(f"\nAverage compound score by country:")
print(df.groupby('country')['compound'].mean().sort_values())

In [ ]:
# Visualize sentiment results
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Geopolitical News Sentiment Analysis', fontsize=16, fontweight='bold')

# 1. Average sentiment by country
avg_sentiment = df.groupby('country')['compound'].mean().sort_values()
colors = ['#d32f2f' if x < -0.1 else '#f57c00' if x < 0 else '#388e3c' for x in avg_sentiment]
avg_sentiment.plot.barh(ax=axes[0,0], color=colors, edgecolor='black')
axes[0,0].set_title('Average Sentiment Score by Country')
axes[0,0].set_xlabel('Compound Sentiment Score')
axes[0,0].axvline(x=0, color='black', linestyle='--', linewidth=0.8)

# 2. Sentiment label distribution by country
sentiment_counts = df.groupby(['country', 'sentiment_label']).size().unstack(fill_value=0)
sentiment_counts.plot.bar(ax=axes[0,1], 
                          color={'NEGATIVE':'#d32f2f','NEUTRAL':'#fbc02d','POSITIVE':'#388e3c'},
                          edgecolor='black')
axes[0,1].set_title('Sentiment Distribution by Country')
axes[0,1].set_xlabel('Country')
axes[0,1].tick_params(axis='x', rotation=45)
axes[0,1].legend(title='Sentiment')

# 3. Sentiment over time
df['date'] = df['published'].dt.date
daily_sentiment = df.groupby(['date','country'])['compound'].mean().reset_index()
for country in targets.keys():
    country_data = daily_sentiment[daily_sentiment['country']==country]
    axes[1,0].plot(country_data['date'], country_data['compound'], 
                   marker='o', markersize=3, label=country)
axes[1,0].set_title('Sentiment Trend Over Time')
axes[1,0].set_xlabel('Date')
axes[1,0].set_ylabel('Compound Score')
axes[1,0].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
axes[1,0].legend(fontsize=8)
axes[1,0].tick_params(axis='x', rotation=45)

# 4. Most negative headlines
most_negative = df.nsmallest(10, 'compound')[['country','title','compound']]
axes[1,1].axis('off')
axes[1,1].set_title('Most Negative Headlines', fontweight='bold')
y_pos = 0.95
for _, row in most_negative.iterrows():
    title_short = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
    axes[1,1].text(0, y_pos, f"[{row['country']}] {title_short}", 
                   fontsize=7, transform=axes[1,1].transAxes, verticalalignment='top')
    axes[1,1].text(0, y_pos-0.04, f"Score: {row['compound']:.3f}", 
                   fontsize=6, color='red', transform=axes[1,1].transAxes)
    y_pos -= 0.09

plt.tight_layout()
plt.savefig(os.path.join(os.path.expanduser("~/Downloads/geopolitical-news-sentiment"), 
            'sentiment_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Dashboard saved.")

In [ ]:
# Export results
output_folder = os.path.expanduser("~/Downloads/geopolitical-news-sentiment")

# Save full results
df.to_csv(os.path.join(output_folder, 'sentiment_results.csv'), index=False)

# Summary statistics
summary = df.groupby('country').agg(
    total_articles=('compound', 'count'),
    avg_sentiment=('compound', 'mean'),
    most_negative=('compound', 'min'),
    most_positive=('compound', 'max'),
    pct_negative=('sentiment_label', lambda x: (x=='NEGATIVE').mean() * 100)
).round(3).sort_values('avg_sentiment')

print("=== GEOPOLITICAL SENTIMENT SUMMARY ===")
print(summary.to_string())
print(f"\nData pulled: {df['published'].min().date()} to {df['published'].max().date()}")
print(f"Total headlines analyzed: {len(df)}")
print("\nResults exported successfully.")